In [ ]:
%config IPCompleter.greedy=True
import os
import random
import copy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model, layers

from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import preprocessing

# region Function Import
from utils import get_train_test_split, get_sample_weights, eval_val_data, eval_te_data, event_seperator, reshape_data, CheckMAE, deterministic_ops



In [ ]:
# Sets all seeds and environment variables to ensure reproducible results across different runs
deterministic_ops(21)

In [ ]:
# IntellEvent is trained on the velocity of the left and right HEEL, ANKLE, and TOE markers 
# using all dimensions (X, Y, Z) resulting in 18 features.
# X = dPlane of movement
# Y = Medial/Lateral
# Z = Up/Down

last_dim = 18

In [ ]:
# Flag for IC or FO; 
# If True - IC model will be trained; If False - FO model will be trained

is_IC = True

In [ ]:
# Read the dataset
dataset = pd.read_pickle("../datasets/BL1.pkl")


# Purpose:
     This pipeline is designed for fine-tuning IntellEvent. 

# Note:
    For training and testing IntellEvent much larger datasets were used - this is just a representation of how the pipelines work
    with the benchmark laboratory datasets! Results vary drastically!!

## Features (Columns):
### Trajectory: 
     Description: Original 3D marker coordinates for the lower extremities.
     Channels: LHEE (X,Y,Z), LTOE (X,Y,Z), LANK (X,Y,Z), RHEE (X,Y,Z), RTOE (X,Y,Z), RANK (X,Y,Z).
     Format: Stacked list/array per trial with shape (18 features, length_trial).
     Directions: 
         X: Anteroposterior (Forward/Backward)
         Y: Vertical (Side-to-Side)
         Z: Mediolateral (Up and Down)

### Velocity:
     Description: First derivative of the 'Trajectory' column.
     Purpose: Standardized input features used as the primary input for the IntellEvent pipeline.

### All_Events:
     Description: Gait event labeled as they are published in the original research for each benchmark laboratory.
     Format: Array of shape (length_trial).
     Mapping: 0 = No Event; 1 = Left IC; 2 = Left FO; 3 = Right IC; 4 = Right FO.

### GRF_Events:
     Description: Ground Truth labels derived strictly from Force Plate data.
     Purpose: Used as the primary target (label) for model training and MAE evaluation to ensure high precision.
     Mapping: Follows the same 0-4 mapping as 'All_Events'.

### Freq_point:
     Description: The sampling frequency (Hz) at which the marker trajectories were recorded.

### Freq_analog:
     Description: The sampling frequency (Hz) of the force plate (analog) data.

### Trial:
     Description: A unique String identifier for the specific walking trial or file name.

### DBid:
     Description: A unique String identifier for the subject (Patient ID).
     Importance: Used for stratification to ensure the same subject does not appear in both Train and Test sets.

### Label:
     Description: Integer identifying the underlying pathology of the subject.
     Reference: Refer to the specific research paper for the pathology-to-integer mapping (e.g., OD, ND, CP).


In [ ]:
# Further function description can be found in 'utils.py': 
# This function is used to split a given dataset into train, test, and validation sets. 
# It splits the data based on the labels present in the dataset and assigns a certain proportion of data to each set.
# Stratifies patients based on Trial ID. All trials from each patient are either in the training, validation or test split.
    # train_data: This DataFrame contains the training data.
    # test_data: This DataFrame contains the test data.
    # train_validation_data: This DataFrame contains the training data for the validation set.
    # test_validation_data: This DataFrame contains the test data for the validation set.

hold_out = 0.3
train_data, test_data, train_validation_data, test_validation_data = get_train_test_split(dataset, hold_out)

In [ ]:
# Mmaps specific gait phases to binary labels (0 or 1). It then adds random padding (5-125 frames) 
# around the first and last detected events to create robust training samples.
# 0 = no Event; 1 = event.

train_grf, train_velocity = event_seperator(copy.deepcopy(train_validation_data['GRF_Events']), copy.deepcopy(train_validation_data['Velocity']), is_IC)
validation_grf, validation_velocity = event_seperator(copy.deepcopy(test_validation_data['GRF_Events']), copy.deepcopy(test_validation_data['Velocity']), is_IC)


In [ ]:
# Further function description in 'utils.py':
# This function reshapes and pads sequences of trajectory and GRF data to the longest input sequence,
# so that they can be used as input for the neural network. The input shape of train_velocity is (features, num_frames).
# The input shape of train_grf is (num_frames).
# Specifically, it pads the sequences with zeros so that all sequences have the same length 
# (the length of the longest sequence), and it reshapes the data to have the shape 
# (num_samples, max_seq_length, num_features) for train_velocity and (num_samples, num_frames) for train_grf, 
# as this is the input format for IntellEvent.  

train_velocity, train_grf = reshape_data(train_velocity, train_grf)
validation_velocity, validation_grf = reshape_data(validation_velocity, validation_grf)

In [ ]:
# samples are HIGHLY imbalanced between no-events and events, therefore we use sample weights to 
# give more weight to rarely-seen events. A ratio of 1:10 (0.1 to 1) was found to be the best ratio.

sample_weights = get_sample_weights(train_grf, [0.1, 1])
train_grf = tf.expand_dims(train_grf.astype(np.float32), axis=-1)
validation_grf = tf.expand_dims(validation_grf.astype(np.float32), axis=-1)

In [ ]:
test_validation_velocity, test_validation_grf = reshape_data(test_validation_data['Velocity'], test_validation_data['GRF_Events'])
val_labels = test_validation_data['Label']

In [ ]:
# Purpose:
    # Implements a transfer learning strategy by loading IntellEvent and 
    # configuring it for further optimization (fine-tuning) on a specific dataset.

# Components:
    # Model Selection (is_IC): 
        # Conditionally loads either the Initial Contact (IC) or Foot Off (FO)  
        # model weights based on the current evaluation target.
    # Trainable Status: 
        # Sets 'pretrained_model.trainable = True' to allow the weights of the existing 
        # bidirectional LSTM and Dense layers to be updated during the new training phase.
    # Compilation:
        # Loss Function: Switches from the standard Binary Focal Crossentropy to 
        # Binary Crossentropy, which can be more stable when fine-tuning with a 
        # limited number of samples.
        # Optimizer: Uses a reduced learning rate (0.0005 * 0.1) to ensure that the 
        # pretrained features are not "washed out" by large weight updates, allowing 
        # for subtle adaptation to the new data.

# Description:
    # This block initializes the fine-tuning phase of the pipeline. By starting with a 
    # model already trained on a large single-center dataset, the system leverages 
    # existing knowledge of gait patterns.
##########################################################################################################################################

if is_IC:
    pretrained_model = tf.keras.models.load_model('../models/SingleCenter_IntellEvent_IC.keras', compile=False)
else:
    pretrained_model = tf.keras.models.load_model('../models/SingleCenter_IntellEvent_FO.keras', compile=False)
    
pretrained_model.trainable = True

pretrained_model.compile(
    # The original IntellEvent-Fine-Tuning uses the BinaryFocalCrossentropy loss function.
    # For testing reasons with low number of samples the BinaryCrossentropy was used here
    #loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=1.5, alpha=0.6),
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.0005*0.1),
)

In [ ]:
# Purpose:
    # Executes the fine-tuning of the pretrained model on the target dataset, balancing 
    # training efficiency with clinical performance metrics.

# Components:
    # EarlyStopping (es): 
        # A regularization callback that monitors 'val_mae'. It stops training if no 
        # improvement is seen for 15 epochs, ensuring the model doesn't overfit to 
        # small datasets. It restores the weights from the best epoch and begins 
        # monitoring after an initial 5-epoch "warm-up" period.
    # CheckMAE (mae_performance): 
        # A custom callback that evaluates and logs the timing error (in frames) for 
        # each specific pathology group at the end of every epoch.
    # model.fit():
        # The training loop execution. It uses a small batch size (8) and sample 
        # weights to ensure the model learns effectively from potentially imbalanced 
        # gait trials.

# Description:
    # This block represents the final training stage where the single-center model 
    # is adapted to multi-center data. By combining standard Keras callbacks with 
    # the custom 'CheckMAE' logic, the pipeline provides real-time visibility into 
    # the model's accuracy across different clinical populations while automatically 
    # preventing training stagnation.
##########################################################################################################################################

es = EarlyStopping(
    monitor='val_mae', 
    patience=15, 
    restore_best_weights=True, 
    verbose=1,
    mode="min",
    start_from_epoch=5
    )

mae_performance = CheckMAE(test_validation_velocity, test_validation_grf, val_labels, is_IC)

history = pretrained_model.fit( 
    train_velocity,
    train_grf,
    validation_data=(validation_velocity, validation_grf),
    callbacks=[mae_performance, es],
    shuffle=True,
    epochs=100,
    batch_size=8,
    sample_weight=sample_weights,
)

In [ ]:
# Purpose:
    # Prepares the unseen test dataset for final evaluation by restructuring temporal sequences 
    # and generating model inferences.

# Components:
    # reshape_data: 
        # Processes the 'Velocity' features and 'GRF_Events' labels into a 3D tensor format 
        # (Samples, Time Steps, Features) compatible with the model's input layer.
    # Metadata Extraction (labels, ids, trials): 
        # Preserves original subject identifiers and pathology labels. This allows the 
        # subsequent evaluation functions to group results by pathology and identify 
        # specific trials where the model may have failed.
    # final_model.predict: 
        # Generates the probabilistic output for the test set. A batch size of 100 is 
        # used to optimize memory efficiency during the forward pass.

# Description:
    # This block represents the transition from model training to final performance 
    # assessment. By extracting subject IDs and trial names alongside the predictions, 
    # the pipeline maintains full traceability, enabling a detailed clinical analysis 
    # of the model's timing accuracy across different patient groups.
##########################################################################################################################################

test_velocity, test_grf = reshape_data(test_data['Velocity'], test_data['GRF_Events'])

labels = test_data['Label']
ids = test_data['DBid']
trials = test_data['Trial']

test_predictions = pretrained_model.predict(test_velocity, batch_size=100)

In [ ]:
# Final evaluation on pathology (label) level
# more details in utils.py
ic_mae_all, ic_list, tp, fn = eval_te_data(labels, ids, trials, test_predictions, test_grf, True)

In [ ]:
# show MAE on patholog (Label) level (Results are in Frames - adapt to ms accordingly, depending on the capturing frequency): 
ic_mae_all

In [ ]:
# Save the results to a pickle file for further processing
# Create a DataFrame with group_choice as the pathology labels
# and the used model for easier plotting in seaborn


# Add which labels/Pathologies are present
group_choice = ["LABEL1", "LABEL2"]
df_results = pd.DataFrame()

for i in range(0,2):
    group_choices = np.full(len(ic_list[i]), group_choice[i]) 
    if i == 0:
        df_results = pd.DataFrame({"ic_list": np.array(ic_list[i]), "Pathology": group_choices, 
                                    "Model": "IntellEvent-Fine-tuning", "Label": "BL1", "TP": np.array(tp[i]), "FP": np.array(fp[i]), "FN": np.array(fn[i])})
    else:
        df_results = pd.concat([df_results, pd.DataFrame({"ic_list": np.array(ic_list[i]), "Pathology": group_choices, 
                                    "Model": "IntellEvent-Fine-tuning", "Label": "BL1", "TP": np.array(tp[i]), "FP": np.array(fp[i]), "FN": np.array(fn[i])})])
df_results.to_pickle("IntellEvent_IC_FT_BL1.pkl")